# LaDe-P Demand Forecasting (Hourly, AOI-Level)

## Goal
Develop an AI model to forecast hourly delivery demand per zone (AOI) using the LaDe (Cainiao) last-mile dataset.

## Forecasting Unit
Each observation represents a unique combination of:

- `city`
- `region_id`
- `aoi_id`
- `hour`

### Target Variable
- `demand_count` — number of deliveries within the specified AOI and hour.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

# Pandas display
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

print("Ready.")
print("Ready.")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Time grouping
TIME_BUCKET = "h"   # Hourly

# Split ratios based on time
TRAIN_RATIO = 0.60
VAL_RATIO   = 0.20
TEST_RATIO  = 0.20

# Lag/rolling configuration
LAGS         = [1, 2, 24, 48, 168]
ROLL_WINDOWS = [24, 168]

# Set DATE_START/DATE_END or MAX_AOIS to None to disable the filter.
DATE_START = None  # inclusive; set to None to keep full range
DATE_END   = None   # inclusive; set to None to keep full range
MAX_AOIS   = 4000            # keep the busiest AOIs; set to None for all

In [9]:
from google.colab import drive
drive.mount('/content/drive')

In [11]:
DATA_PATH   = "/content/drive/MyDrive/Senior Project/LaDe-Dataset/Dataset.csv"
OUTPUT_PATH = "/content/drive/MyDrive/Senior Project/LaDe-Dataset/lade_hourly_features.csv"

df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
df_raw.head()

# Section 4 — Column Selection & Data Dictionary

## Key Columns Used

### 1. Identifiers
- `city` — City name or code  
- `region_id` — Administrative or operational region identifier  
- `aoi_id` — Area of Interest (delivery zone) identifier  
- `aoi_type` — Type/category of AOI  

These define the spatial unit of forecasting.

---

### 2. Time
- `ds` — Date code in `MMDD` format (e.g., `0925` for September 25th)
- `accept_time` — Timestamp in `MM-DD HH:MM:SS` format when the order was accepted

We prepend the data year (`2022`) to `accept_time` to produce a full `accept_dt` datetime.

---

### 3. Target Variable (Derived)
- `demand_count` — Number of orders per `(city, region_id, aoi_id, hour)`

Computed by counting `order_id` occurrences per group after hourly bucketing.

In [12]:
# ── Data Quality Checks ───────────────────────────────────────────────────────

key_cols = ["city", "region_id", "aoi_id", "aoi_type", "accept_time", "ds"]
print("Missing values in key columns:")
print(df_raw[key_cols].isna().sum())

print()
print("Unique cities:  ", df_raw["city"].nunique())
print("Unique regions: ", df_raw["region_id"].nunique())
print("Unique AOIs:    ", df_raw["aoi_id"].nunique())
print("Unique couriers:", df_raw["courier_id"].nunique())

print()
dup_orders = df_raw["order_id"].duplicated().sum()
print("Duplicate order_id count:", dup_orders)

# Section 6 — Time Parsing

## Timestamp Construction

The dataset provides:
- `ds` — date code in `MMDD` format (e.g., `0925`)
- `accept_time` — timestamp in `MM-DD HH:MM:SS` format (e.g., `09-25 08:08:00`)

We prepend the data year (`2022`) to `accept_time` to build a full datetime:

```
accept_time="09-25 08:08:00"  →  "2022-09-25 08:08:00"
```

## Hourly Bucketing

Each timestamp is floored to the nearest hour:
```
2022-09-25 08:08:00  →  2022-09-25 08:00:00
```

## Demand Definition
Demand is defined using `accept_time` — when the order enters the system.  
This reflects incoming workload, making forecasts suitable for dispatch planning and courier allocation.

In [13]:
# ── Timestamp Parsing (prepend year to accept_time: MM-DD HH:MM:SS → YYYY-MM-DD HH:MM:SS) ─

DATA_YEAR = "2022"

df = df_raw.copy()

# accept_time format is "MM-DD HH:MM:SS" — prepend year to get full datetime
df["accept_dt"] = pd.to_datetime(
    DATA_YEAR + "-" + df["accept_time"].astype(str),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("Unparsed accept_time rows:", df["accept_dt"].isna().sum())
df = df.dropna(subset=["accept_dt"]).copy()

# Optional date-range filtering
if DATE_START is not None:
    df = df[df["accept_dt"] >= DATE_START]
if DATE_END is not None:
    end_exclusive = pd.to_datetime(DATE_END) + pd.Timedelta(days=1)
    df = df[df["accept_dt"] < end_exclusive]

print("Time range after filtering:", df["accept_dt"].min(), "→", df["accept_dt"].max())
df[["ds", "accept_time", "accept_dt"]].head()

# Section 8 — Target Definition via Aggregation Rule

## Hourly Demand Definition

Hourly demand is defined as:

```
demand_count(city, region_id, aoi_id, hour)
= number of orders whose accept_time falls within that hour
```

---

## Aggregation Logic

After parsing timestamps and creating hourly buckets (`bucket_hour`), we compute demand using a group-by aggregation.

### Group Keys
- `city`, `region_id`, `aoi_id`, `aoi_type`, `bucket_hour`

### Aggregation Operation
- Count number of rows (orders) per group → `demand_count`

In [14]:
# ── Hourly Aggregation ────────────────────────────────────────────────────────

df["bucket_hour"] = df["accept_dt"].dt.floor(TIME_BUCKET)

demand_agg = (
    df.groupby(["city", "region_id", "aoi_id", "aoi_type", "bucket_hour"], as_index=False)
      .agg(demand_count=("order_id", "count"))
      .sort_values(["city", "region_id", "aoi_id", "bucket_hour"])
)

print("Aggregated shape:", demand_agg.shape)

# ── Demand Distribution (sanity check) ───────────────────────────────────────
print("\nDemand count distribution (non-zero hours only):")
print("  Min:   ", demand_agg["demand_count"].min())
print("  Max:   ", demand_agg["demand_count"].max())
print("  Mean:  ", round(demand_agg["demand_count"].mean(), 3))
print("  Median:", demand_agg["demand_count"].median())
print()
print(demand_agg["demand_count"].quantile([0.25, 0.5, 0.75, 0.9, 0.99]))

demand_agg.head()

# Section 10 — Why We Must Add Zero-Demand Hours

## The Problem

If we only keep hours where at least one order exists, the dataset excludes periods with no activity.

This creates a biased training signal:
- The model only observes **positive demand**
- It never learns what "no demand" looks like
- Forecasts become systematically overestimated

---

## The Solution

We construct a **complete hourly timeline** for every AOI and fill missing hours with `demand_count = 0`.

---

## Performance Note

Rather than scanning `demand_agg` on every loop iteration (slow), we pre-group it into a dictionary keyed by `(city, region_id, aoi_id, aoi_type)`.  
Each AOI lookup is then O(1) instead of O(n).

In [15]:
# ── AOI Selection ─────────────────────────────────────────────────────────────

if MAX_AOIS is not None:
    aoi_totals = (
        demand_agg.groupby(["city", "region_id", "aoi_id", "aoi_type"])["demand_count"]
        .sum()
        .sort_values(ascending=False)
        .head(MAX_AOIS)
        .reset_index()
    )
    aoi_keys = aoi_totals[["city", "region_id", "aoi_id", "aoi_type"]]
else:
    aoi_keys = demand_agg[["city", "region_id", "aoi_id", "aoi_type"]].drop_duplicates().reset_index(drop=True)

# Global hourly range
start_hour = demand_agg["bucket_hour"].min()
end_hour   = demand_agg["bucket_hour"].max()
full_hours = pd.date_range(start=start_hour, end=end_hour, freq=TIME_BUCKET)

print(f"AOIs selected:          {len(aoi_keys)}")
print(f"Hours per AOI:          {len(full_hours)}")
print(f"Expected total rows:    {len(aoi_keys) * len(full_hours):,}")

# ── Pre-group demand_agg for O(1) AOI lookup (key optimisation) ───────────────
# Instead of scanning all rows per AOI in the loop, we group once here.
agg_grouped = {
    key: grp[["bucket_hour", "demand_count"]].set_index("bucket_hour")
    for key, grp in demand_agg.groupby(["city", "region_id", "aoi_id", "aoi_type"])
}

print(f"\nPre-grouped AOI lookup table built: {len(agg_grouped)} entries")

In [16]:
# ── Grid Builder (zero-fill) ──────────────────────────────────────────────────

def build_aoi_grid(agg_grouped, key_row, full_hours):
    """
    Build a complete hourly time series for one AOI.
    Missing hours are filled with demand_count = 0.
    Uses O(1) dict lookup instead of scanning the full dataframe.
    """
    key = (key_row["city"], key_row["region_id"], key_row["aoi_id"], key_row["aoi_type"])

    g = agg_grouped.get(key, pd.DataFrame(columns=["demand_count"]))
    g = g.reindex(full_hours)
    g["demand_count"] = g["demand_count"].fillna(0).astype("int32")  # int32 avoids int16 overflow risk

    g = g.reset_index().rename(columns={"index": "bucket_hour"})
    g["city"]      = key_row["city"]
    g["region_id"] = key_row["region_id"]
    g["aoi_id"]    = key_row["aoi_id"]
    g["aoi_type"]  = key_row["aoi_type"]

    return g[["city", "region_id", "aoi_id", "aoi_type", "bucket_hour", "demand_count"]]

In [17]:
# ── Feature Engineering ───────────────────────────────────────────────────────

def add_time_and_lag_features(df_aoi):
    """
    Add calendar features, lag features, and rolling mean features.
    Input is already sorted by bucket_hour from build_aoi_grid — no re-sort needed.
    """
    # Calendar features
    df_aoi["hour"]  = df_aoi["bucket_hour"].dt.hour.astype("int8")
    df_aoi["dow"]   = df_aoi["bucket_hour"].dt.dayofweek.astype("int8")
    df_aoi["month"] = df_aoi["bucket_hour"].dt.month.astype("int8")

    # Weekend flag (standard Sat-Sun weekend — applies to China/Jilin data)
    df_aoi["is_weekend"] = df_aoi["dow"].isin([5, 6]).astype("int8")

    # Lag features — past demand at fixed offsets
    for lag in LAGS:
        df_aoi[f"lag_{lag}"] = df_aoi["demand_count"].shift(lag)

    # Rolling mean features — shift(1) ensures no leakage from current hour
    roll_24, roll_168 = ROLL_WINDOWS
    df_aoi["roll_24_mean"]  = df_aoi["demand_count"].shift(1).rolling(roll_24,  min_periods=roll_24).mean()
    df_aoi["roll_168_mean"] = df_aoi["demand_count"].shift(1).rolling(roll_168, min_periods=roll_168).mean()

    # Drop rows where lag/rolling features are NaN (start of each AOI's series)
    feature_cols = [f"lag_{lag}" for lag in LAGS] + ["roll_24_mean", "roll_168_mean"]
    df_aoi = df_aoi.dropna(subset=feature_cols)

    return df_aoi

In [ ]:
# ── Vectorized Processing (no loop) ──────────────────────────────────────────

# Step 1: Build complete grid for ALL AOIs at once (cross join)
aoi_hours = aoi_keys.assign(_k=1).merge(
    pd.DataFrame({"bucket_hour": full_hours, "_k": 1}), on="_k"
).drop(columns="_k")

# Step 2: Left-join actual demand → zero-fill missing hours
model_df = aoi_hours.merge(
    demand_agg[["city", "region_id", "aoi_id", "aoi_type", "bucket_hour", "demand_count"]],
    on=["city", "region_id", "aoi_id", "aoi_type", "bucket_hour"],
    how="left"
)
model_df["demand_count"] = model_df["demand_count"].fillna(0).astype("int32")
model_df = model_df.sort_values(["city", "region_id", "aoi_id", "bucket_hour"]).reset_index(drop=True)
print(f"Grid shape after zero-fill: {model_df.shape}")

# Step 3: Calendar features
model_df["hour"]       = model_df["bucket_hour"].dt.hour.astype("int8")
model_df["dow"]        = model_df["bucket_hour"].dt.dayofweek.astype("int8")
model_df["month"]      = model_df["bucket_hour"].dt.month.astype("int8")
model_df["is_weekend"] = model_df["dow"].isin([5, 6]).astype("int8")

# Step 4: Lag + rolling features grouped by AOI (no bleed between AOIs)
roll_24, roll_168 = ROLL_WINDOWS

def add_features(g):
    for lag in LAGS:
        g[f"lag_{lag}"] = g["demand_count"].shift(lag)
    s = g["demand_count"].shift(1)
    g["roll_24_mean"]  = s.rolling(roll_24,  min_periods=roll_24).mean()
    g["roll_168_mean"] = s.rolling(roll_168, min_periods=roll_168).mean()
    return g

model_df = model_df.groupby(["city", "region_id", "aoi_id"], group_keys=False).apply(add_features, include_groups=False)

# Step 5: Drop NaN rows (start of each AOI's series)
feature_cols = [f"lag_{lag}" for lag in LAGS] + ["roll_24_mean", "roll_168_mean"]
model_df = model_df.dropna(subset=feature_cols).reset_index(drop=True)

# Step 6: Final column order + save
feature_cols_order = (
    ["city", "region_id", "aoi_id", "aoi_type", "bucket_hour",
     "hour", "dow", "month", "is_weekend", "demand_count"]
    + [f"lag_{lag}" for lag in LAGS]
    + ["roll_24_mean", "roll_168_mean"]
)
model_df = model_df[feature_cols_order]
model_df.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved: {OUTPUT_PATH}")
print(f"Final dataset shape: {model_df.shape}")

In [ ]:
# ── Sanity Check ──────────────────────────────────────────────────────────────

preview = pd.read_csv(OUTPUT_PATH, nrows=5)
preview

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────

print("=" * 50)
print("Feature generation complete.")
print("=" * 50)
print(f"  AOIs processed : {len(aoi_keys)}")
print(f"  Total rows     : {len(model_df):,}")
print(f"  Columns        : {list(model_df.columns)}")
print(f"  Date range     : {model_df['bucket_hour'].min()} → {model_df['bucket_hour'].max()}")
print(f"  Output file    : {OUTPUT_PATH}")

In [ ]:
aoi_totals = demand_agg.groupby("aoi_id")["demand_count"].sum().sort_values(ascending=False)
coverage = aoi_totals.cumsum() / aoi_totals.sum()

n_70 = (coverage <= 0.70).sum()
n_80 = (coverage <= 0.80).sum()
n_90 = (coverage <= 0.90).sum()
n_100 = (coverage <= 1.00).sum()


print(f"AOIs needed for 70% demand coverage: {n_70}")
print(f"AOIs needed for 80% demand coverage: {n_80}")
print(f"AOIs needed for 90% demand coverage: {n_90}")
print(f"AOIs needed for 100% demand coverage: {n_100}")
